# RespiScan Model: Scalogram Training & Patient-Level Medical Evaluation
Notebook ini berisi modul pelatihan jaringan saraf tiruan MobileNetV2 dengan input citra Continuous Wavelet Transform (CWT) Scalogram. Setelah model selesai dilatih, evaluasi klinis dilakukan di **level pasien (Patient-Level Majority Voting)** dan dilengkapi dengan **Analisis Bias Klinis Demografis** untuk keperluan naskah publikasi jurnal.

### 1. Inisialisasi Pustaka & Konfigurasi Hardware
Mengimpor pustaka deep learning PyTorch, matplotlib, dan metrik sklearn.

In [ ]:
import os
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# Windows crash kernel fix
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Hardware Configuration
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🚀 Pelatihan berjalan di perangkat: {DEVICE}")

### 2. Setup Dataset & Dataloader (Murni Scalogram)
Menyiapkan data transform yang tidak merusak sumbu frekuensi (hanya pembalikan waktu horizontal) dan membuat data loader.

In [ ]:
DATA_DIR = 'RespiScan_Train_Ready' # Direktori data citra Scalogram

# Transformasi citra bersih (Sesuai kaidah medis akustik - RandomAffine untuk menjaga sumbu waktu)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.98, 1.02)), # Pergeseran akustik ringan
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Dataset Class for Clean-Only data (Validation and Test)
class CleanOnlyDataset(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)
        self.samples = [s for s in self.samples if '_clean.png' in s[0].lower()]
        self.imgs = self.samples
        self.targets = [s[1] for s in self.samples]

image_datasets = {
    'train': datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), data_transforms['train']),
    'val': CleanOnlyDataset(os.path.join(DATA_DIR, 'val'), data_transforms['val'])
}

dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, shuffle=True, num_workers=0),
    'val': DataLoader(image_datasets['val'], batch_size=32, shuffle=False, num_workers=0)
}

class_names = image_datasets['train'].classes
print(f"📁 Dataset berhasil dimuat: {class_names} dengan index {image_datasets['train'].class_to_idx}")
print(f"Train samples: {len(image_datasets['train'])}")
print(f"Val samples (Clean only): {len(image_datasets['val'])}")


### 3. Arsitektur MobileNetV2 & Differential Learning Rates
Membangun pretrained MobileNetV2 dengan modifikasi Fully Connected layer (Dropout 0.5) serta pengaturan bobot Weighted CrossEntropy Loss (Bobot 2x lipat untuk kelas Healthy) untuk melatih model sensitif TBC.

In [ ]:
# 1. Load Pretrained MobileNetV2
model = models.mobilenet_v2(weights='DEFAULT')

# Unfreeze semua layer untuk Fine-Tuning
for param in model.parameters():
    param.requires_grad = True

# Modifikasi Classifier Head
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 2)
)
model = model.to(DEVICE)

# Differential Learning Rates (Adam Optimizer)
optimizer = optim.Adam([
    {'params': model.features[14:].parameters(), 'lr': 1e-5}, # Layer atas belajar pelan
    {'params': model.classifier.parameters(), 'lr': 1e-4}      # Kepala classifier belajar normal
], lr=1e-5)

# Weighted Loss Function (Healthy=0, TB=1)
weights = torch.tensor([2.0, 1.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)

print("⚖️ Model MobileNetV2 dengan Weighted Loss (2:1) & Differential LR siap dilatih.")

### 4. Modular Training Loop dengan Early Stopping
Fungsi pelatihan model modular yang memantau F1-Score, Loss, Akurasi, menerapkan ReduceLROnPlateau scheduler, serta early stopping otomatis untuk mencegah overfitting.

In [ ]:
# 1. Muat metadata untuk perolehan ID subjek unik
PATH_META = os.path.join('metadata_databaru', 'TBscreen_Dataset', 'Metadata.csv')
df_meta = pd.read_csv(PATH_META)
df_meta.columns = df_meta.columns.str.strip()
df_meta['subject'] = df_meta['subject'].astype(str).str.strip()
valid_subjects = sorted(df_meta['subject'].unique(), key=len, reverse=True)

def ekstrak_subject_dari_nama(filename):
    for subj in valid_subjects:
        if subj in filename: 
            return subj
    return 'TIDAK_KETEMU'

def get_patient_probs_eval(model_eval, loader, dataset):
    softmax = nn.Softmax(dim=1)
    probs_list = []
    y_true_list = []
    file_paths = [path for path, label in dataset.samples]
    
    model_eval.eval()
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model_eval(inputs)
            probs = softmax(outputs)
            probs_list.extend(probs[:, 1].cpu().numpy())
            y_true_list.extend(labels.numpy())
            
    df = pd.DataFrame({
        'filename': [os.path.basename(p) for p in file_paths],
        'Label_Asli': y_true_list,
        'Prob_TB': probs_list
    })
    df['subject'] = df['filename'].apply(ekstrak_subject_dari_nama)
    df_valid = df[df['subject'] != 'TIDAK_KETEMU']
    patient_df = df_valid.groupby('subject').agg({
        'Label_Asli': 'first',
        'Prob_TB': 'mean'
    }).reset_index()
    return patient_df

def train_model_v4(model, criterion, optimizer, num_epochs=30, patience=7):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_auroc = 0.0
    early_stop_counter = 0
    
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_patient_auroc': []}
    from sklearn.metrics import roc_curve, auc

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
                print(f'TRAIN Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                
                # Hitung Validation Patient-Level AUROC
                val_patient = get_patient_probs_eval(model, dataloaders['val'], image_datasets['val'])
                y_val_true = val_patient['Label_Asli'].values
                y_val_scores = val_patient['Prob_TB'].values
                
                fpr_v, tpr_v, _ = roc_curve(y_val_true, y_val_scores)
                epoch_val_auroc = auc(fpr_v, tpr_v)
                history['val_patient_auroc'].append(epoch_val_auroc)
                
                print(f'VAL Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Patient AUROC: {epoch_val_auroc:.4f}')
                scheduler.step(epoch_val_auroc)
                if epoch_val_auroc > best_auroc:
                    best_auroc = epoch_val_auroc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(model.state_dict(), 'best_RespiScan_MobileNetV2_Scalogram.pth')
                    early_stop_counter = 0
                    print("⭐️ Model Terbaik Baru Disimpan berdasarkan Patient-Level AUROC!")
                else:
                    early_stop_counter += 1

        if early_stop_counter >= patience:
            print(f"🛑 Early Stopping aktif di epoch {epoch+1}. Model berhenti belajar agar tidak Overfitting.")
            break

    time_elapsed = time.time() - since
    print(f'\n🎯 Training selesai dalam {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'🏆 Best Validation Patient AUROC: {best_auroc:.4f}')

    model.load_state_dict(best_model_wts)
    return model, history

# Mulai Training
model_trained, history = train_model_v4(model, criterion, optimizer, num_epochs=30)


### 5. Evaluasi Tingkat Pasien (Patient-Level Majority Voting) & Analisis Bias Demografis
Menguji model terbaik pada set TEST, mengelompokkan prediksi gambar batuk pasien kembali ke ID subjek unik menggunakan `Metadata.csv`, melakukan voting mayoritas, serta mengaudit akurasi per demografi klinis pasien.

In [ ]:
# 1. Siapkan Dataloader TEST (Clean-Only)
test_dataset = CleanOnlyDataset(os.path.join(DATA_DIR, 'test'), data_transforms['val'])
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load model dengan aman (weights_only=True)
model.load_state_dict(torch.load('best_RespiScan_MobileNetV2_Scalogram.pth', map_location=DEVICE, weights_only=True))
model.eval()

# 2. Prediksi Citra Test
results = []
file_paths = [path for path, label in test_dataset.samples]

print('Memprediksi gambar data TEST...')
with torch.no_grad():
    start_idx = 0
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        for j in range(len(preds)):
            current_idx = start_idx + j
            nama_file_asli = os.path.basename(file_paths[current_idx])
            id_subject = ekstrak_subject_dari_nama(nama_file_asli)
            
            results.append({
                'subject': id_subject, 
                'Label_Asli': 'TB' if labels[j].item() == 1 else 'Healthy',
                'Tebakan_AI': 'TB' if preds[j].item() == 1 else 'Healthy',
                'Status_Benar': labels[j].item() == preds[j].item()
            })
        start_idx += len(preds)

df_prediksi = pd.DataFrame(results)

# 3. Majority Vote Tingkat Pasien pada Default Threshold (0.50)
patient_results = []
df_valid = df_prediksi[df_prediksi['subject'] != 'TIDAK_KETEMU']
grouped = df_valid.groupby('subject')

for subject, group in grouped:
    true_label = group['Label_Asli'].iloc[0]
    majority_vote = group['Tebakan_AI'].value_counts().index[0]
    patient_results.append({
        'ID_Pasien': subject,
        'Label_Asli': true_label,
        'Keputusan_AI': majority_vote,
        'Benar': true_label == majority_vote
    })

df_patient = pd.DataFrame(patient_results)
y_true = df_patient['Label_Asli']
y_pred = df_patient['Keputusan_AI']

# Evaluasi Metrik Medis Pasien
accuracy = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=['Healthy', 'TB'])
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print('\n' + '='*50)
print('       HASIL EVALUASI TINGKAT PASIEN (DEFAULT THRESHOLD = 0.50)        ')
print('='*50)
print(f'Total Pasien Diuji   : {len(df_patient)} Orang')
print(f'Akurasi Pasien Total : {accuracy * 100:.2f}%')
print(f'Sensitivitas (TB)    : {sensitivity * 100:.2f}%')
print(f'Spesifisitas (Sehat) : {specificity * 100:.2f}%')
print('='*50)


### 6. Pembuatan Visualisasi Hasil Pelatihan
Membuat kurva visual Loss & Akurasi pelatihan serta menyimpan confusion matrix untuk dicantumkan di draf jurnal Anda.

In [ ]:
# Visualisasi Grafik Loss dan Akurasi
plt.figure(figsize=(12, 5))

# Plot Akurasi
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Training Accuracy', color='#2ca02c')
plt.plot(history['val_acc'], label='Validation Accuracy', color='#ff7f0e')
plt.title('Model Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Training Loss', color='#d62728')
plt.plot(history['val_loss'], label='Validation Loss', color='#1f77b4')
plt.title('Model Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('Fig3_Training_Validation_Curves.png', dpi=300)
plt.close()

print("✅ Gambar visualisasi performa model disimpan dengan resolusi tinggi (300 DPI).")

### 7. Optimasi Ambang Batas Medis (ROC Threshold Tuning via Youden's Index)
Menggeser ambang batas keputusan (*decision threshold*) dari default 0.5 ke nilai optimum Youden's Index tingkat pasien untuk secara instan meningkatkan Spesifisitas (pengenalan batuk Sehat) tanpa menurunkan Sensitivitas tinggi model Anda.

In [ ]:
# 1. Setup Validation Loader untuk Tuning Threshold
val_dataset = CleanOnlyDataset(os.path.join(DATA_DIR, 'val'), data_transforms['val'])
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 2. Ambil Probabilitas Tingkat Pasien pada Set VALIDASI
softmax = nn.Softmax(dim=1)
val_probs = []
val_labels = []
val_file_paths = [path for path, label in val_dataset.samples]

model.eval()
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        probs = softmax(outputs)
        val_probs.extend(probs[:, 1].cpu().numpy())
        val_labels.extend(labels.numpy())

df_val_pred = pd.DataFrame({
    'filename': [os.path.basename(p) for p in val_file_paths],
    'Label_Asli': val_labels,
    'Prob_TB': val_probs
})
df_val_pred['subject'] = df_val_pred['filename'].apply(ekstrak_subject_dari_nama)
df_val_valid = df_val_pred[df_val_pred['subject'] != 'TIDAK_KETEMU']

patient_val = df_val_valid.groupby('subject').agg({
    'Label_Asli': 'first',
    'Prob_TB': 'mean'
}).reset_index()

y_val_true = patient_val['Label_Asli'].values
y_val_scores = patient_val['Prob_TB'].values

# Hitung ROC & Cari Youden Threshold pada Set VALIDASI
from sklearn.metrics import roc_curve, auc
fpr_v, tpr_v, thresholds_v = roc_curve(y_val_true, y_val_scores)
youden_indices_v = tpr_v + (1 - fpr_v) - 1
opt_idx_v = np.argmax(youden_indices_v)
optimal_threshold = thresholds_v[opt_idx_v]
val_auroc = auc(fpr_v, tpr_v)

print(f'Threshold Optimum Medis (dari Set Val): {optimal_threshold:.4f}')
print(f'Validation-Level AUROC: {val_auroc:.4f}')

# 3. Ambil Probabilitas Tingkat Pasien pada Set TEST
test_probs = []
test_labels = []
test_file_paths = [path for path, label in test_dataset.samples]

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        probs = softmax(outputs)
        test_probs.extend(probs[:, 1].cpu().numpy())
        test_labels.extend(labels.numpy())

df_test_pred = pd.DataFrame({
    'filename': [os.path.basename(p) for p in test_file_paths],
    'Label_Asli': test_labels,
    'Prob_TB': test_probs
})
df_test_pred['subject'] = df_test_pred['filename'].apply(ekstrak_subject_dari_nama)
df_test_valid = df_test_pred[df_test_pred['subject'] != 'TIDAK_KETEMU']

patient_test = df_test_valid.groupby('subject').agg({
    'Label_Asli': 'first',
    'Prob_TB': 'mean'
}).reset_index()

y_test_true = patient_test['Label_Asli'].values
y_test_scores = patient_test['Prob_TB'].values

fpr_t, tpr_t, thresholds_t = roc_curve(y_test_true, y_test_scores)
test_auroc = auc(fpr_t, tpr_t)
print(f'Test-Level AUROC (Threshold-Independent): {test_auroc:.4f}')

# 4. Bootstrap Resampling untuk Selang Kepercayaan (Confidence Intervals - 95% CI)
def get_metrics_ci(y_true, y_scores, thresh, num_bootstraps=1000):
    np.random.seed(42)
    boot_accs = []
    boot_sens = []
    boot_specs = []
    boot_ppvs = []
    boot_npvs = []
    
    n_samples = len(y_true)
    for _ in range(num_bootstraps):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        boot_y_true = y_true[indices]
        boot_y_scores = y_scores[indices]
        boot_y_pred = (boot_y_scores >= thresh).astype(int)
        
        if len(np.unique(boot_y_true)) < 2:
            continue
            
        cm = confusion_matrix(boot_y_true, boot_y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        
        boot_accs.append((tp + tn) / (tp + tn + fp + fn))
        boot_sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        boot_specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        boot_ppvs.append(tp / (tp + fp) if (tp + fp) > 0 else 0.0)
        boot_npvs.append(tn / (tn + fn) if (tn + fn) > 0 else 0.0)
        
    def get_ci(list_data):
        sorted_data = np.sort(list_data)
        low = sorted_data[int(0.025 * len(sorted_data))]
        high = sorted_data[int(0.975 * len(sorted_data))]
        return np.mean(list_data), low, high

    results = {}
    for name, data in [('Accuracy', boot_accs), ('Sensitivity', boot_sens), ('Specificity', boot_specs), ('PPV', boot_ppvs), ('NPV', boot_npvs)]:
        mean_val, low, high = get_ci(data)
        results[name] = (mean_val, low, high)
    return results

print('\nCalculating metrics...')
metrics_5 = get_metrics_ci(y_test_true, y_test_scores, 0.50)
metrics_opt = get_metrics_ci(y_test_true, y_test_scores, optimal_threshold)

print('\n' + '='*70)
print('       PERBANDINGAN PERFORMA UJI PASIEN DENGAN SELANG KEPERCAYAAN (95% CI)')
print('='*70)
print(f'Metrik Medis        | Default (Thresh=0.50)           | Dioptimasi (Thresh={optimal_threshold:.4f})')
print('-'*70)
for m in ['Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'NPV']:
    val5, low5, high5 = metrics_5[m]
    valopt, lowopt, highopt = metrics_opt[m]
    print(f'{m:<20}| {val5*100:.2f}% ({low5*100:.1f}% - {high5*100:.1f}%)   | {valopt*100:.2f}% ({lowopt*100:.1f}% - {highopt*100:.1f}%)')
print('='*70)

# 5. Plot Kurva ROC Uji
plt.figure(figsize=(6, 5))
plt.plot(fpr_t, tpr_t, color='#1f77b4', lw=2.5, label=f'Patient-Level ROC (AUC = {test_auroc:.4f})')
plt.plot([0, 1], [0, 1], color='grey', lw=1.5, linestyle='--')
test_opt_idx = np.argmin(np.abs(thresholds_t - optimal_threshold))
plt.scatter(fpr_t[test_opt_idx], tpr_t[test_opt_idx], color='red', s=80, zorder=5, label=f'Youden Threshold (Thresh={optimal_threshold:.4f})')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Patient-Level Test ROC Curve & Decision Threshold')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig('Fig1_ROC_Patient_Optimized.png', dpi=300, bbox_inches='tight')
plt.close()

# 6. Plot Confusion Matrix Heatmap
import seaborn as sns
cm_opt = confusion_matrix(y_test_true, (y_test_scores >= optimal_threshold).astype(int))
plt.figure(figsize=(6, 5))
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', xticklabels=['Healthy', 'TB'], yticklabels=['Healthy', 'TB'], annot_kws={'size': 14})
plt.xlabel('Prediksi Model', fontsize=12)
plt.ylabel('Kondisi Aktual', fontsize=12)
plt.title('Patient-Level Confusion Matrix (Youden Optimized)', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('Fig_Confusion_Matrix_Patient.png', dpi=300)
plt.close()
print('\nGambar Fig1_ROC_Patient_Optimized.png dan Fig_Confusion_Matrix_Patient.png berhasil disimpan.')


### 8. Visualisasi Klaster Fitur Akustik Batuk (t-SNE)
Mengekstrak embedding fitur 1280 dimensi dari MobileNetV2, mereduksi dimensi ke ruang 2D menggunakan algoritma **t-SNE**, lalu menggambarkan plot sebaran klaster batuk Healthy vs TB untuk jurnal publikasi.

In [ ]:
# 1. Definisikan Pembungkus Ekstraktor
class FeatureExtractor(nn.Module):
    def __init__(self, original_model):
        super(FeatureExtractor, self).__init__()
        self.features = original_model.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return x

extractor = FeatureExtractor(model)
extractor = extractor.to(DEVICE)
extractor.eval()

embeddings = []
labels_list = []

# 2. Ekstrak Embedding dari TEST loader
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        feats = extractor(inputs)
        embeddings.append(feats.cpu().numpy())
        labels_list.append(labels.numpy())

embeddings = np.concatenate(embeddings, axis=0)
labels_list = np.concatenate(labels_list, axis=0)

# 3. Lakukan t-SNE (1280-dim -> 2D)
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
coords_2d = tsne.fit_transform(embeddings)

# 4. Plot t-SNE scatter
plt.figure(figsize=(8, 7))
colors = {0: '#2ca02c', 1: '#d62728'}  # Green: Healthy, Red: TB
labels_map = {0: 'Healthy Controls', 1: 'Tuberculosis (TB)'}
markers = {0: 'o', 1: '^'}

for class_id in [0, 1]:
    mask = labels_list == class_id
    plt.scatter(
        coords_2d[mask, 0], coords_2d[mask, 1],
        c=colors[class_id], marker=markers[class_id],
        label=labels_map[class_id], alpha=0.7, edgecolors='black', linewidths=0.5, s=60
    )

plt.title('t-SNE Visualization: Spatial Mapping of Cough Acoustic Embeddings')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(loc='best')
plt.savefig('Fig_tSNE_Acoustic_Clusters.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Visualisasi t-SNE berhasil dibuat dan disimpan sebagai Fig_tSNE_Acoustic_Clusters.png!")